# Séance 2 — Pandas : manipulation & nettoyage

**Decision problem:** how do bad dates, missing values, duplicates, and inconsistent categories create bad decisions?

Official topic preserved: pandas manipulation and cleaning. The output is a clean analytical table used by the rest of the course.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
SNAPSHOT = "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "snapshots" / SNAPSHOT).exists():
        ROOT = candidate
        break
OUT = ROOT / "outputs"
for d in [OUT / "bloc1", OUT / "bloc2", OUT / "bloc3", OUT / "final_product"]:
    d.mkdir(parents=True, exist_ok=True)
DATA = ROOT / "data" / "snapshots"

def load_raw_trends():
    p = DATA / "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
    df = pd.read_csv(p)
    df["date"] = pd.to_datetime(df["date"])
    for c in ["chatgpt", "iphone", "meteo"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.sort_values("date").reset_index(drop=True)

def load_clean_long():
    p = OUT / "bloc1" / "clean_trends_long.csv"
    if p.exists():
        df = pd.read_csv(p, parse_dates=["date"])
    else:
        raw = load_raw_trends()
        df = raw.melt(id_vars="date", var_name="signal", value_name="interest")
    return df.sort_values(["date", "signal"]).reset_index(drop=True)

In [ ]:
raw = load_raw_trends()
quality = pd.DataFrame({"dtype": raw.dtypes.astype(str), "missing": raw.isna().sum(), "unique": raw.nunique()})
quality

In [ ]:
signals = ["chatgpt", "iphone", "meteo"]
clean_wide = raw.dropna(subset=["date"]).sort_values("date").drop_duplicates("date")
clean_wide[signals] = clean_wide[signals].ffill().bfill()
clean_long = clean_wide.melt(id_vars="date", value_vars=signals, var_name="signal", value_name="interest")
clean_long["interest"] = clean_long["interest"].clip(0, 100)

assert clean_long["interest"].between(0, 100).all()
assert clean_long[["date", "signal", "interest"]].notna().all().all()

clean_wide.to_csv(OUT / "bloc1" / "clean_trends_wide.csv", index=False)
clean_long.to_csv(OUT / "bloc1" / "clean_trends_long.csv", index=False)
quality.to_csv(OUT / "bloc1" / "data_quality_report.csv")
print("Saved clean datasets to", OUT / "bloc1")

## Practical exercise

Add one new validation rule that would prevent a bad management dashboard.

## Conclusion

The pipeline now has a trusted input table for analysis, modeling, and BI.